Topics: Data Modelling and Search Models
* Question anwering over pandas data frames

# Part 1: Data Preparation

In [ ]:
import requests
import pandas as pd
import json

# Get table data
def get_table_data(url):

    response = requests.get(url)

    # store all tables in the tables list
    tables = []
    geometry_data = response.json()
    # loop through the dataset and convert tabular data to pandas dataframes
    for doc in geometry_data:
        table = pd.DataFrame(doc["data"], columns=doc["header"])
        tables.append(table)

    return tables, geometry_data

tables, geometry_data = get_table_data("https://raw.githubusercontent.com/aurioldegbelo/sis2026/refs/heads/main/vector_data/data_table.json")
print(len(tables))
print(type(tables[0]))
tables[0]

2
<class 'pandas.core.frame.DataFrame'>


,id,type,name,population,area,state_capital,geometry
0,s01,State,Nordrhein-Westfalen,18139116,34110.26,Düsseldorf,nrw_geom
1,s02,State,Hessen,6391360,21114.94,Wiesbaden,hessen_geom
2,s03,State,Rheinland-Pfalz,4159150,19854.21,Mainz,rlp_geom
3,s04,State,Niedersachsen,8140242,47709.82,Hannover,niedersachsen_geom


In [ ]:
tables[1]

,id,type,touches,url
0,nrw_geom,Geometry,"[hessen_geom, rlp_geom, niedersachsen_geom]",https://github.com/aurioldegbelo/sis2026/blob/...
1,hessen_geom,Geometry,"[nrw_geom, rlp_geom, niedersachsen_geom]",https://github.com/aurioldegbelo/sis2026/blob/...
2,rlp_geom,Geometry,"[nrw_geom, hessen_geom]",https://github.com/aurioldegbelo/sis2026/blob/...
3,niedersachsen_geom,Geometry,"[hessen_geom, nrw_geom]",https://github.com/aurioldegbelo/sis2026/blob/...


# Part 2: QA using [Langchain](https://docs.langchain.com/build-overview)


In [ ]:
!pip install --quiet langchain langchain_openai langchain_experimental

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import os
import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

Enter your OpenAI API key: ··········


One data frame

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_experimental.tools import PythonAstREPLTool

df = tables[0]

# Initialize the language model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Create a Python execution tool
python_tool = PythonAstREPLTool(locals={"df": df}) # Ast (Abstract Syntax Tree); REPL: Read-Eval-Print Loop
# useful to execute arbitrary Python code: https://sj-langchain.readthedocs.io/en/latest/tools/langchain.tools.python.tool.PythonAstREPLTool.html

# Define the agent instructions
system_prompt = f"""
You are an experienced Python data analyst.

You have access to a pandas DataFrame called df.

DataFrame columns:
{list(df.columns)}

DataFrame preview:
{df.head().to_string()}

Instructions:
- Answer questions using the actual DataFrame.
- Use pandas operations to calculate results.
- Never invent data or statistics.
- Execute Python code using the available Python tool.
- Explain calculations and results clearly.
- If a column does not exist, explain that to the user.

For every task, explain the step-by-step process that led to your answer:
1. What you tried to do (e.g. which cell/row/table you looked into exactly).
2. Which tool you used and why.
3. Explain what you learned from the tool result.

Do not skip steps.

Your final answer should have the format
Answer
# Summary of the steps
1. ...
2. ...
3. ...
"""

# Create the agent
agent = create_agent(
    model=llm,
    tools=[python_tool],
    system_prompt=system_prompt) # Add debug=True if you want to see intermediate steps

# Ask questions
question1 = "How many states in the database?"
question2 = "How many geometries in the the database?"
question3 = "What is the population of Hessen?"
question4 = "What is the area of Hessen?"
question5 = "What is the capital of Hessen?"
question6 = "What is the geometry of Hessen?"
question7 = "What are the geometries of Hessen and Niedersachsen?"
question8 = "What is the url of the geometry of Hessen?"

question9 = "generate higlights about population in the dataset"
question10 = "say what interesting spatial patterns the dataset offers"

response = agent.invoke({"messages": [
            {"role": "user", "content": question10}]})

# 8. Display the answer
print(response["messages"][-1].content)

I have calculated the population density for each state by dividing the population by the area. Here are the results:

| Name                   | Population | Area (sq km) | Population Density (people/sq km) |
|------------------------|------------|---------------|-------------------------------------|
| Nordrhein-Westfalen    | 18,139,116 | 34,110.26     | 531.78                              |
| Hessen                  | 6,391,360  | 21,114.94     | 302.69                              |
| Rheinland-Pfalz        | 4,159,150  | 19,854.21     | 209.48                              |
| Niedersachsen          | 8,140,242  | 47,709.82     | 170.62                              |

# Summary of the steps
1. I calculated the population density for each state by dividing the population by the area using pandas operations.
2. I created a new column in the DataFrame called 'population_density' to store these values.
3. From the results, I learned that Nordrhein-Westfalen has the highest population 

In [ ]:
print(system_prompt)


You are an experienced Python data analyst.

You have access to a pandas DataFrame called df.

DataFrame columns:
['id', 'type', 'name', 'population', 'area', 'state_capital', 'geometry']

DataFrame preview:
    id   type                 name  population      area state_capital            geometry
0  s01  State  Nordrhein-Westfalen    18139116  34110.26    Düsseldorf            nrw_geom
1  s02  State               Hessen     6391360  21114.94     Wiesbaden         hessen_geom
2  s03  State      Rheinland-Pfalz     4159150  19854.21         Mainz            rlp_geom
3  s04  State        Niedersachsen     8140242  47709.82      Hannover  niedersachsen_geom

Instructions:
- Answer questions using the actual DataFrame.
- Use pandas operations to calculate results.
- Never invent data or statistics.
- Execute Python code using the available Python tool.
- Explain calculations and results clearly.
- If a column does not exist, explain that to the user.

For every task, explain the step-by-s

Several dataframes

In [ ]:
tables

[    id   type                 name  population      area state_capital  \
 0  s01  State  Nordrhein-Westfalen    18139116  34110.26    Düsseldorf   
 1  s02  State               Hessen     6391360  21114.94     Wiesbaden   
 2  s03  State      Rheinland-Pfalz     4159150  19854.21         Mainz   
 3  s04  State        Niedersachsen     8140242  47709.82      Hannover   
 
              geometry  
 0            nrw_geom  
 1         hessen_geom  
 2            rlp_geom  
 3  niedersachsen_geom  ,
                    id      type                                      touches  \
 0            nrw_geom  Geometry  [hessen_geom, rlp_geom, niedersachsen_geom]   
 1         hessen_geom  Geometry     [nrw_geom, rlp_geom, niedersachsen_geom]   
 2            rlp_geom  Geometry                      [nrw_geom, hessen_geom]   
 3  niedersachsen_geom  Geometry                      [hessen_geom, nrw_geom]   
 
                                                  url  
 0  https://github.com/aurioldegbe

In [ ]:
all_df = tables

python_tool2 = PythonAstREPLTool(locals={"df": all_df})

system_prompt2 = f"""
You are an experienced Python data analyst.

You have access to a pandas DataFrame called df.

all_df:{all_df}

Instructions:
- Answer questions using the actual DataFrame.
- Use pandas operations to calculate results.
- Never invent data or statistics.
- Execute Python code using the available Python tool.
- Explain calculations and results clearly.
- If a column does not exist, explain that to the user.

For every task, explain the step-by-step process that led to your answer:
1. What you tried to do (e.g. which cell/row/table you looked into exactly).
2. Which tool you used and why.
3. Explain what you learned from the tool result.

Do not skip steps.

Your final answer should have the format
Answer
# Summary of the steps
1. ...
2. ...
3. ...
"""

agent2 = create_agent(
    model=llm,
    tools=[python_tool2],
    system_prompt=system_prompt2)

In [ ]:
print(system_prompt2)


You are an experienced Python data analyst.

You have access to a pandas DataFrame called df.

all_df:[    id   type                 name  population      area state_capital  \
0  s01  State  Nordrhein-Westfalen    18139116  34110.26    Düsseldorf   
1  s02  State               Hessen     6391360  21114.94     Wiesbaden   
2  s03  State      Rheinland-Pfalz     4159150  19854.21         Mainz   
3  s04  State        Niedersachsen     8140242  47709.82      Hannover   

             geometry  population_density  
0            nrw_geom          531.778884  
1         hessen_geom          302.693732  
2            rlp_geom          209.484538  
3  niedersachsen_geom          170.619843  ,                    id      type                                      touches  \
0            nrw_geom  Geometry  [hessen_geom, rlp_geom, niedersachsen_geom]   
1         hessen_geom  Geometry     [nrw_geom, rlp_geom, niedersachsen_geom]   
2            rlp_geom  Geometry                      [nrw_geom, 

In [ ]:
question1 = "How many states in the database?"
question2 = "How many geometries in the the database?"
question3 = "What is the population of Hessen?"
question4 = "What is the area of Hessen?"
question5 = "What is the capital of Hessen?"
question6 = "What is the geometry of Hessen?"
question7 = "What are the geometries of Hessen and Niedersachsen?"
question8 = "What is the url of the geometry of Hessen?"

question9 = "Which state(s) touch Hessen?"
question10 = "What is the ulr of nrw_geom?"

response = agent2.invoke({"messages": [
            {"role": "user", "content": question10}]})

print(response["messages"][-1].content)

The URL for `nrw_geom` is: 

**https://github.com/aurioldegbelo/sis2026/blob/...**

# Summary of the steps
1. I created a DataFrame structure based on the provided data to simulate the original DataFrame.
2. I filtered the `geometry_df` DataFrame to find the row where the `id` is `nrw_geom` and extracted the corresponding `url`.
3. I successfully retrieved the URL for `nrw_geom`.


# Available Models

In [ ]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
models = client.models.list()

# Filter and print models (e.g., looking for GPT models)
for model in models.data:
    if "gpt" in model.id:
        print(model.id)

gpt-live-transcribe
gpt-5.6-luna
gpt-realtime-2
gpt-5-search-api
gpt-5.5-2026-04-23
gpt-5-mini
gpt-5-mini-2025-08-07
gpt-4o-mini-transcribe-2025-12-15
gpt-5-pro-2025-10-06
gpt-5.6-sol
gpt-4o-2024-11-20
gpt-5.1
gpt-5-pro
gpt-realtime-whisper
gpt-4-turbo-2024-04-09
gpt-realtime-2.1-mini
gpt-realtime-2025-08-28
gpt-5.4-2026-03-05
gpt-3.5-turbo-0125
gpt-5-nano
gpt-5.1-codex
gpt-5.4-mini
gpt-5.4-pro
gpt-realtime-mini-2025-12-15
gpt-4o-mini-tts
gpt-5.1-codex-max
gpt-4o-mini-search-preview
gpt-5.4-pro-2026-03-05
gpt-realtime-translate
gpt-realtime
gpt-realtime-1.5
gpt-5.2-pro
gpt-5-chat-latest
gpt-4-turbo
gpt-4o-2024-08-06
gpt-4o-mini-search-preview-2025-03-11
gpt-5-2025-08-07
gpt-5.2-pro-2025-12-11
gpt-audio-mini-2025-10-06
gpt-5.5-pro-2026-04-23
gpt-4o-transcribe-diarize
gpt-5-codex
gpt-5.1-2025-11-13
gpt-4-0613
gpt-4o-search-preview
gpt-5.4
gpt-4o-search-preview-2025-03-11
gpt-4.1-nano
gpt-image-2
gpt-3.5-turbo-16k
gpt-realtime-2.1
gpt-4.1
gpt-5.5
gpt-5.3-chat-latest
gpt-5.2-2025-12-11
cha